### Import packages

In [ ]:
import torch
import pandas as pd
import numpy as np
import re
import string
import matplotlib.pyplot as plt
from datasets import Dataset
import wandb
from transformers import AutoTokenizer
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, f1_score

### Load dataset

In [ ]:
data = pd.read_csv('/kaggle/input/ag-news-classification-dataset/train.csv')
test = pd.read_csv('/kaggle/input/ag-news-classification-dataset/test.csv')
data.head() 

#### some explorations

In [ ]:
data.shape, test.shape

In [ ]:
data['Class Index'].unique(), test['Class Index'].unique()

In [ ]:
data['Class Index'] = data['Class Index']-1
test['Class Index'] = test['Class Index']-1

In [ ]:
data['Class Index'].unique(), test['Class Index'].unique()

### AG News Labels:
### 0: World
### 1: Sports
### 2: Business
### 3: Sci/Tech


In [ ]:
data['Class Index'].value_counts()

In [ ]:
test['Class Index'].value_counts()

In [ ]:
data.isnull().sum()

In [ ]:
data['Description'][2]

In [ ]:
data['Description'][3]

In [ ]:
slash = 0
slash_n = 0
douple_space = 0
for text in data['Description']:
    if '\\' in text:
        slash+=1
    if '\n'in data['Description']:
        slash_n+=1
    if r'\s+' in data['Description']:
        douple_space+=1

print('\\ in ',slash, 'samples')
print('\\n in ',slash_n, 'samples')
print('douple space in ',douple_space, 'samples')


### Data preprocessing

In [ ]:
# Concatenate Title + Description
data['text'] = data['Title'] + " " + data['Description']
test['text'] = test['Title'] + " " + test['Description']

In [ ]:
data = data.drop(columns=['Title', 'Description'])
test = test.drop(columns=['Title', 'Description'])

In [ ]:
data.head()

In [ ]:
def preprocessing(text):
    text = text.replace('\\', ' ')
    text = re.sub(f"[{re.escape(string.punctuation)}]", " ", text)
    text = re.sub(r'\s+', ' ', text)
    text = text.lower()
    return text

In [ ]:
data['text'] = data['text'].apply(lambda x: preprocessing(x))
test['text'] = test['text'].apply(lambda x: preprocessing(x))

data['text'][3]

In [ ]:
# Convert Pandas DataFrame to HuggingFace Dataset
dataset = Dataset.from_pandas(data)
testing = Dataset.from_pandas(test)

In [ ]:
# Tokenization
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

def tokenization(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=512)

encoded_dataset = dataset.map(tokenization, batched=True)
encoded_testing = testing.map(tokenization, batched=True)

In [ ]:
# Remove old text columns, keep model inputs
encoded_dataset = encoded_dataset.rename_column('Class Index', 'labels')
encoded_testing = encoded_testing.rename_column('Class Index', 'labels')

encoded_dataset.set_format('torch')
encoded_testing.set_format('torch')

### Load model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Load pre-trained DistilBERT model for classification
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=4)
model.to(device)

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",       
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    save_total_limit=1,
    logging_dir='./logs',
    logging_steps=50,
    # report_to=None,                  
    fp16=True if torch.cuda.is_available() else False
)


In [ ]:
# Define evaluation metrics
def compute_metrics(pred):
    logits, labels = pred
    predictions = torch.argmax(torch.tensor(logits), dim=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="weighted")
    return {"accuracy": acc, "f1": f1}

### Model training

In [ ]:
# wandb.login(key='******')
wandb.init(project="AG news classification") 

In [ ]:
# Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset,
    eval_dataset=encoded_testing,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)
trainer.train() 

In [ ]:
# Save final model
trainer.save_model("./agnews_distilbert_model")
tokenizer.save_pretrained("./agnews_distilbert_model")


### Evaluation

In [ ]:
results = trainer.evaluate()
print("Final Evaluation Results:", results)

### Visualization

In [ ]:
# Training logs
training_logs = trainer.state.log_history

# Extract training loss and validation accuracy
train_loss = [log["loss"] for log in training_logs if "loss" in log]
eval_acc = [log["eval_accuracy"] for log in training_logs if "eval_accuracy" in log]

In [ ]:
plt.figure(figsize=(12,5))

# Training Loss
plt.subplot(1,2,1)
plt.plot(train_loss, label="Training Loss")
plt.title("Training Loss Curve")
plt.xlabel("Steps")
plt.ylabel("Loss")
plt.legend()

# Validation Accuracy
plt.subplot(1,2,2)
plt.plot(eval_acc, label="Validation Accuracy", color='orange')
plt.title("Validation Accuracy Curve")
plt.xlabel("Evaluation Steps")
plt.ylabel("Accuracy")
plt.legend()

plt.show()

### Demo

In [ ]:
def predict(texts):
    model.eval()
    inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt", max_length=512)
    inputs = {key: val.to(device) for key, val in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    preds = torch.argmax(outputs.logits, dim=-1)
    return preds

In [ ]:
# Test sample predictions
sample_texts = [
    "The government has announced a new policy for education.",
    "The Lakers won their last match by a huge margin!",
    "Apple released a new iPhone model today.",
    "NASA plans a new mission to Mars next year."
]

predictions = predict(sample_texts)
print("Predicted Labels:", predictions)

### AG News Labels:
### 0: World
### 1: Sports
### 2: Business
### 3: Sci/Tech


In [ ]:
for text, label in zip(sample_texts, predictions):
    print(f"Text: {text}\nPredicted Label: {label.item()}\n")